In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pathogen_atlas_v3").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


# Phase 3 · Jev disease gate and domain features

Phase 3 first asks whether each selected species–abstract pair provides human disease evidence. Pairs at or above the configured 0.75 confidence threshold then receive all applicable features for their organism group in one additional Jev request. Successful outputs resume only when the run, abstract hash, model, question version, and feature profile still match.

In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from pathogen_atlas_v3.common.config import selected_organism_groups
from pathogen_atlas_v3.common.io import load_yaml
from pathogen_atlas_v3.decisions.jev import openrouter_key_available
from pathogen_atlas_v3.pipelines.phase3_decisions import phase3_preflight, run_phase3

CONFIG_PATH = "configs/default.yaml"
CONFIG = load_yaml(CONFIG_PATH)
GROUPS = selected_organism_groups(CONFIG)
print("Selected groups:", GROUPS)
print("Model:", CONFIG["decisions"]["model"])
print("Disease threshold:", CONFIG["decisions"]["pathogen_threshold"])
print("Feature yes threshold:", CONFIG["decisions"]["feature_yes_threshold"])
print("OpenRouter key available in environment or Keychain:", openrouter_key_available())
print("Concurrency:", CONFIG["decisions"]["max_workers"], "| progress/checkpoint interval:", CONFIG["logging"]["progress_every"])
print("Decision ledger:", CONFIG["paths"]["decision_ledger"])
print("Raw Jev responses:", CONFIG["paths"]["raw_responses"])

Selected groups: ['viruses']
Model: typesafe/jev-1.14
Disease threshold: 0.75
Feature yes threshold: 0.75
OpenRouter key available in environment or Keychain: True
Concurrency: 32 | progress/checkpoint interval: 100
Decision ledger: data/03_decisions/parsed/decision_ledger.parquet
Raw Jev responses: data/03_decisions/raw_responses/jev.jsonl


## Preflight · no Jev calls

Review the selected pair count, pending gate and feature calls, estimated tokens, and configured USD estimate before running. The Jev runner prints per-stage progress, failure counts, ETA, and remaining OpenRouter balance.

In [3]:
PREFLIGHT = phase3_preflight(CONFIG_PATH)
display(pd.DataFrame([PREFLIGHT]))

,eligible_species_pmid_pairs,cached_gate_pairs,pending_gate_calls,estimated_feature_calls,cached_feature_pairs,estimated_input_tokens,estimated_output_tokens,estimated_cost_usd,cost_note
0,187325,0,187325,93662,0,1348734000,140493500,56.646828,Estimate uses configured token assumptions and...


## Run or resume the complete Jev scoring run

`MAX_CALLS = None` runs every eligible gate and feature decision for the selected groups. Finish Phase 2 first so the published abstracts and species–PMID matches are complete, then review the preflight estimate and available OpenRouter balance. The decision ledger checkpoints successful work, so reruns resume without repeating matching successful decisions.

In [ ]:
RUN_PHASE = True
MAX_CALLS = None  # full selected-domain gate and feature scoring
LOG_PROGRESS = True

if RUN_PHASE:
    if not openrouter_key_available():
        raise RuntimeError("Set OPENROUTER_API_KEY or add the existing OpenRouter Keychain item.")
    DECISIONS = run_phase3(CONFIG_PATH, max_calls=MAX_CALLS, log=LOG_PROGRESS)
    display({"gate_rows": len(DECISIONS["gate"]), "feature_rows": len(DECISIONS["features"]), "selected_groups": GROUPS})
else:
    print("Preflight only. Set RUN_PHASE=True to run or resume Jev decisions.")

## Release policy

The probability thresholds remain provisional. Phase 4 marks candidate evidence as accepted only when `decisions.release_evaluation_approved` is true after labelled evaluation.